# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [ ]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [ ]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '15:36:04', 'date': '2026-09-19', 'day': 'Saturday'}


## Part 2: Tool Registry

Manage available tools.

In [ ]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

#Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [ ]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        # Simplified: pattern matching for demo
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            # Extract math expression from query
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer
        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer

# Test the agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))
print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Agent Tests:
Q: What is 25 * 4?
A: Error: invalid syntax (<string>, line 0)

Q: What time is it?
A: It's 15:38:58 on 2026-09-19 (Saturday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [ ]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 15:39:28 on 2026-09-19 (Saturday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [4]:
from datetime import datetime
from typing import Dict, List, Any, Callable
import re

# --- 1. Re-defining Helper Classes with Prioritized Intent Detection ---

class ToolRegistry:
    '''Manages available tools for the agent.'''
    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

class SimpleReActAgent:
    '''Simple agent that can use tools.'''
    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take (prioritizing specific tools over general symbol match).'''
        query_lower = query.lower()
        # Prioritize 'note' and 'habit' to avoid '-' getting caught in calculator
        if 'note' in query_lower:
            return 'use_note_taker'
        elif 'habit' in query_lower:
            return 'use_habit_tracker'
        elif any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}
        elif action == 'use_time':
            return self.registry.get_tool('get_time')()
        elif action == 'use_note_taker':
            note_content = query.replace('take a note', '').replace('note:', '').strip()
            return self.registry.get_tool('note_taker')(note_content)
        elif action == 'use_habit_tracker':
            habit_name = query.replace('track habit', '').replace('habit:', '').strip()
            return self.registry.get_tool('habit_tracker')(habit_name)
        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        if 'result' in observation:
            return f"The result is {observation['result']}"
        elif 'time' in observation:
            return f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'status' in observation:
            return observation['message']
        elif 'total_completions' in observation:
            return observation['message']
        elif 'error' in observation:
            return f"Error: {observation['error']}"
        else:
            return "I can help with calculations, time, notes, and habits."

class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''
    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        def default_calc(expression: str): return {'result': eval(expression)}
        def default_time(): return {'time': datetime.now().strftime('%H:%M:%S'), 'date': datetime.now().strftime('%Y-%m-%d'), 'day': datetime.now().strftime('%A')}
        self.registry.register('calculator', default_calc, 'Calculate math')
        self.registry.register('get_time', default_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# --- 2. Custom Tools Implementation ---

notes_db = []

def note_taker(note_text: str) -> dict:
    '''Saves a new note to the notes collection.'''
    notes_db.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'note': note_text
    })
    return {'status': 'success', 'message': f"Saved note: '{note_text}'"}

habits_db = {}

def habit_tracker(habit_name: str) -> dict:
    '''Logs or increments completion of a daily habit.'''
    habits_db[habit_name] = habits_db.get(habit_name, 0) + 1
    return {
        'habit': habit_name,
        'total_completions': habits_db[habit_name],
        'message': f"Logged habit '{habit_name}'. Completed {habits_db[habit_name]} times."
    }

# --- 3. Digital Twin Setup and Registration ---

twin = DigitalTwinWithTools('My Assistant')
twin.add_tool('note_taker', note_taker, 'Saves and logs a user-provided note')
twin.add_tool('habit_tracker', habit_tracker, 'Logs daily habit completion and counts occurrences')

# Let's test the tools directly and via twin chat
print("Direct Tool Tests:")
print(note_taker("Remember to review python agents today!"))
print(habit_tracker("Drink 8 glasses of water"))
print(habit_tracker("Drink 8 glasses of water"))

print("\nDigital Twin Chat Tests with tools:")
print("User: note: Buy fresh groceries")
print("Twin:", twin.chat("note: Buy fresh groceries"))
print("\nUser: habit: Go for a 30-minute run")
print("Twin:", twin.chat("habit: Go for a 30-minute run"))


Direct Tool Tests:
{'status': 'success', 'message': "Saved note: 'Remember to review python agents today!'"}
{'habit': 'Drink 8 glasses of water', 'total_completions': 1, 'message': "Logged habit 'Drink 8 glasses of water'. Completed 1 times."}
{'habit': 'Drink 8 glasses of water', 'total_completions': 2, 'message': "Logged habit 'Drink 8 glasses of water'. Completed 2 times."}

Digital Twin Chat Tests with tools:
User: note: Buy fresh groceries
Twin: Saved note: 'Buy fresh groceries'

User: habit: Go for a 30-minute run
Twin: Logged habit 'Go for a 30-minute run'. Completed 1 times.


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI

## Week 3.2: Testing Your AI

In this section, we will write automated test cases to evaluate the reliability of your digital twin and its custom tools. We will cover:
- Unit tests for tools (`note_taker`, `habit_tracker`)
- Integration tests for agent intent classification
- Evaluation scripts to run multiple scenarios automatically

In [6]:
# Week 3.2 Test Suite Setup
def run_test_suite():
    print("--- Starting Week 3.2 Agent Test Suite ---")

    # Test 1: Note Taker
    initial_count = len(notes_db)
    note_taker("Test note")
    assert len(notes_db) == initial_count + 1, "Note taker failed to append note."
    print("Test 1: Note Taker tool verification passed.")

    # Test 2: Habit Tracker
    test_habit = "Read a book"
    habit_tracker(test_habit)
    assert habits_db[test_habit] >= 1, "Habit tracker failed to log completion."
    print(" Test 2: Habit Tracker tool verification passed.")

    # Test 3: Agent Intent Classification (ReAct)
    twin_test = DigitalTwinWithTools('Test Twin')
    twin_test.add_tool('note_taker', note_taker, 'Saves notes')
    twin_test.add_tool('habit_tracker', habit_tracker, 'Tracks habits')

    # Verify that query containing 'note' routes to note_taker
    assert twin_test.agent.think("take a note: buy milk") == "use_note_taker", "Agent routing failed for note taking."
    # Verify that query containing 'habit' routes to habit_tracker
    assert twin_test.agent.think("habit: jog 10km") == "use_habit_tracker", "Agent routing failed for habit tracking."
    print(" Test 3: Agent Routing Intent classification passed.")

    print("\nAll initial test cases passed successfully!")

run_test_suite()

--- Starting Week 3.2 Agent Test Suite ---
Test 1: Note Taker tool verification passed.
 Test 2: Habit Tracker tool verification passed.
 Test 3: Agent Routing Intent classification passed.

All initial test cases passed successfully!
